
═══════════════════════════════════════════════════════════════════════════
📌 Notebook 1: Coleta de Dados do IBGE
═══════════════════════════════════════════════════════════════════════════

Neste notebook vamos:
✅ Entender as fontes de dados disponíveis
✅ Acessar APIs do IBGE (SIDRA)
✅ Baixar dados municipais
✅ Consolidar em um único dataset
✅ Fazer primeira inspeção dos dados

⏱️ Tempo estimado: 15-20 minutos

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 1: Importações
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import requests
import json
from pathlib import Path
from datetime import datetime
import time

print("🚀 PROJETO: CLUSTERIZAÇÃO DE MUNICÍPIOS BRASILEIROS")
print("="*80)
print(f"📅 Data de execução: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print("="*80)
print()
print("✅ Bibliotecas importadas!")


🚀 PROJETO: CLUSTERIZAÇÃO DE MUNICÍPIOS BRASILEIROS
📅 Data de execução: 23/01/2026 07:17:25

✅ Bibliotecas importadas!


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 2: Entendendo as fontes de dados
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📚 FONTES DE DADOS DO IBGE")
print("="*80)
print()

print("🌐 Principais fontes:")
print()
print("1️⃣  API SIDRA (Sistema IBGE de Recuperação Automática)")
print("   📍 URL: https://apisidra.ibge.gov.br/")
print("   📊 Dados: Tabelas agregadas de diversos indicadores")
print("   ✅ Vantagem: API gratuita, dados oficiais e atualizados")
print()

print("2️⃣  IBGE Cidades")
print("   📍 URL: https://cidades.ibge.gov.br/")
print("   📊 Dados: Panorama completo de cada município")
print("   ⚠️  Limitação: Não tem API oficial, precisa web scraping")
print()

print("3️⃣  API de Localidades")
print("   📍 URL: https://servicodados.ibge.gov.br/api/docs/localidades")
print("   📊 Dados: Informações geográficas e códigos dos municípios")
print("   ✅ Vantagem: API REST simples e estável")
print()

print("📌 NESTE PROJETO USAREMOS:")
print("   → API de Localidades (lista de municípios)")
print("   → API SIDRA (indicadores socioeconômicos)")
print()


📚 FONTES DE DADOS DO IBGE

🌐 Principais fontes:

1️⃣  API SIDRA (Sistema IBGE de Recuperação Automática)
   📍 URL: https://apisidra.ibge.gov.br/
   📊 Dados: Tabelas agregadas de diversos indicadores
   ✅ Vantagem: API gratuita, dados oficiais e atualizados

2️⃣  IBGE Cidades
   📍 URL: https://cidades.ibge.gov.br/
   📊 Dados: Panorama completo de cada município
   ⚠️  Limitação: Não tem API oficial, precisa web scraping

3️⃣  API de Localidades
   📍 URL: https://servicodados.ibge.gov.br/api/docs/localidades
   📊 Dados: Informações geográficas e códigos dos municípios
   ✅ Vantagem: API REST simples e estável

📌 NESTE PROJETO USAREMOS:
   → API de Localidades (lista de municípios)
   → API SIDRA (indicadores socioeconômicos)



In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 3: Função para buscar municípios
# ═══════════════════════════════════════════════════════════════════════════

print("="*80)
print("📍 COLETANDO LISTA DE MUNICÍPIOS")
print("="*80)
print()

def get_municipios():
    print("🔄 Buscando municípios via API...")

    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    municipios = response.json()

    data = []
    for m in municipios:
        # tenta UF via microrregiao
        uf_obj = (
            (m.get("microrregiao") or {})
            .get("mesorregiao") or {}
        ).get("UF")

        # fallback: tenta UF via regiao-imediata
        if not uf_obj:
            uf_obj = (
                (m.get("regiao-imediata") or {})
                .get("regiao-intermediaria") or {}
            ).get("UF")

        uf_obj = uf_obj or {}
        regiao_obj = uf_obj.get("regiao") or {}

        data.append({
            "codigo_ibge": m.get("id"),
            "nome_municipio": m.get("nome"),
            "uf": uf_obj.get("sigla"),
            "nome_uf": uf_obj.get("nome"),
            "regiao": regiao_obj.get("nome"),
        })

    df = pd.DataFrame(data)

    print(f"✅ {len(df):,} municípios coletados!")
    print("\n⚠️ Linhas sem UF:", df["uf"].isna().sum())
    print("\n📊 Distribuição por região (inclui NaN):")
    print(df["regiao"].value_counts(dropna=False).to_string())

    return df

# Executar coleta
df_municipios = get_municipios()

# Exibir amostra
print()
print("📋 Amostra dos dados:")
print(df_municipios.head(10))

📍 COLETANDO LISTA DE MUNICÍPIOS

🔄 Buscando municípios via API...
✅ 5,571 municípios coletados!

⚠️ Linhas sem UF: 0

📊 Distribuição por região (inclui NaN):
regiao
Nordeste        1794
Sudeste         1668
Sul             1191
Centro-Oeste     468
Norte            450

📋 Amostra dos dados:
   codigo_ibge         nome_municipio  uf   nome_uf regiao
0      1100015  Alta Floresta D'Oeste  RO  Rondônia  Norte
1      1100023              Ariquemes  RO  Rondônia  Norte
2      1100031                 Cabixi  RO  Rondônia  Norte
3      1100049                 Cacoal  RO  Rondônia  Norte
4      1100056             Cerejeiras  RO  Rondônia  Norte
5      1100064      Colorado do Oeste  RO  Rondônia  Norte
6      1100072             Corumbiara  RO  Rondônia  Norte
7      1100080          Costa Marques  RO  Rondônia  Norte
8      1100098        Espigão D'Oeste  RO  Rondônia  Norte
9      1100106          Guajará-Mirim  RO  Rondônia  Norte


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 4: Função para buscar população
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("👥 COLETANDO DADOS DE POPULAÇÃO")
print("="*80)
print()

print("📌 Tabela SIDRA: 6579 - População residente estimada")
print("📅 Ano mais recente disponível")
print()

def get_populacao(ano=2025):
    """
    Busca população estimada dos municípios
    
    Args:
        ano: Ano de referência (default: 2022)
    
    Retorna:
        DataFrame com código do município e população
    """
    print(f"🔄 Buscando população (ano {ano})...")
    
    # URL da API SIDRA - Tabela 6579
    url = f"https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/9324/p/{ano}"
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        data = response.json()
        
        # Converter para DataFrame
        df = pd.DataFrame(data[1:], columns=data[0])  # Primeira linha são headers
        
        # Limpar e processar
        df = df[['D1C', 'V']].copy()  # D3C = código município, V = valor
        df.columns = ['codigo_ibge', 'populacao']
        
        # Converter tipos
        df['codigo_ibge'] = pd.to_numeric(df['codigo_ibge'], errors='coerce')
        df['populacao'] = pd.to_numeric(df['populacao'], errors='coerce')
        
        # Remover valores nulos
        df = df.dropna()
        
        print(f"✅ População de {len(df):,} municípios coletada!")
        print()
        print("📊 Estatísticas:")
        print(f"   População total: {df['populacao'].sum():,.0f} habitantes")
        print(f"   Maior município: {df['populacao'].max():,.0f} habitantes")
        print(f"   Menor município: {df['populacao'].min():,.0f} habitantes")
        print(f"   Média: {df['populacao'].mean():,.0f} habitantes")
        print(f"   Mediana: {df['populacao'].median():,.0f} habitantes")
        
        return df
        
    except Exception as e:
        print(f"❌ Erro ao buscar população: {e}")
        return None

# Executar coleta
df_populacao = get_populacao(2025)

# Exibir amostra
if df_populacao is not None:
    print()
    print("📋 Amostra dos dados:")
    print(df_populacao.head(10))


👥 COLETANDO DADOS DE POPULAÇÃO

📌 Tabela SIDRA: 6579 - População residente estimada
📅 Ano mais recente disponível

🔄 Buscando população (ano 2025)...
✅ População de 5,571 municípios coletada!

📊 Estatísticas:
   População total: 213,421,037 habitantes
   Maior município: 11,904,961 habitantes
   Menor município: 856 habitantes
   Média: 38,309 habitantes
   Mediana: 11,355 habitantes

📋 Amostra dos dados:
   codigo_ibge  populacao
0      1100015      22787
1      1100023     109170
2      1100031       5664
3      1100049      98280
4      1100056      16966
5      1100064      16508
6      1100072       7968
7      1100080      13510
8      1100098      32842
9      1100106      43594


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 5: Função para buscar PIB
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("💰 COLETANDO DADOS DE PIB")
print("="*80)
print()

print("📌 Tabela SIDRA: 5938 - Produto Interno Bruto dos Municípios")
print("📅 Ano mais recente disponível (geralmente 2 anos defasado)")
print()

def get_pib(ano=2021):
    """
    Busca PIB municipal
    
    Args:
        ano: Ano de referência (default: 2021 - último disponível)
    
    Retorna:
        DataFrame com código do município e PIB
    """
    print(f"🔄 Buscando PIB (ano {ano})...")
    
    # URL da API SIDRA - Tabela 5938
    # Variável 37 = PIB a preços correntes
    url = f"https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/37/p/{ano}"
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        data = response.json()
        
        # Converter para DataFrame
        df = pd.DataFrame(data[1:], columns=data[0])
        
        # Limpar e processar
        df = df[['D1C', 'V']].copy()
        df.columns = ['codigo_ibge', 'pib_mil_reais']
        
        # Converter tipos
        df['codigo_ibge'] = pd.to_numeric(df['codigo_ibge'], errors='coerce')
        df['pib_mil_reais'] = pd.to_numeric(df['pib_mil_reais'], errors='coerce')
        
        # Converter de mil reais para milhões
        df['pib_milhoes'] = df['pib_mil_reais'] / 1000
        
        # Remover valores nulos
        df = df.dropna()
        
        print(f"✅ PIB de {len(df):,} municípios coletado!")
        print()
        print("📊 Estatísticas:")
        print(f"   PIB total: R$ {df['pib_milhoes'].sum():,.0f} milhões")
        print(f"   Maior PIB: R$ {df['pib_milhoes'].max():,.0f} milhões")
        print(f"   Menor PIB: R$ {df['pib_milhoes'].min():,.0f} milhões")
        print(f"   Média: R$ {df['pib_milhoes'].mean():,.0f} milhões")
        print(f"   Mediana: R$ {df['pib_milhoes'].median():,.0f} milhões")
        
        return df[['codigo_ibge', 'pib_milhoes']]
        
    except Exception as e:
        print(f"❌ Erro ao buscar PIB: {e}")
        return None

# Executar coleta
df_pib = get_pib(2021)

# Exibir amostra
if df_pib is not None:
    print()
    print("📋 Amostra dos dados:")
    print(df_pib.head(10))


💰 COLETANDO DADOS DE PIB

📌 Tabela SIDRA: 5938 - Produto Interno Bruto dos Municípios
📅 Ano mais recente disponível (geralmente 2 anos defasado)

🔄 Buscando PIB (ano 2021)...
✅ PIB de 5,570 municípios coletado!

📊 Estatísticas:
   PIB total: R$ 9,012,142 milhões
   Maior PIB: R$ 828,981 milhões
   Menor PIB: R$ 18 milhões
   Média: R$ 1,618 milhões
   Mediana: R$ 253 milhões

📋 Amostra dos dados:
   codigo_ibge  pib_milhoes
0      1100015      734.467
1      1100023     3211.294
2      1100031      238.414
3      1100049     2792.506
4      1100056      743.062
5      1100064      424.846
6      1100072      396.740
7      1100080      316.672
8      1100098      773.372
9      1100106     1054.255


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 6: Buscar área territorial
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🗺️ COLETANDO ÁREA TERRITORIAL")
print("="*80)
print()

def get_area():
    """
    Busca área territorial dos municípios
    
    Retorna:
        DataFrame com código do município e área em km²
    """
    print("🔄 Buscando área territorial...")
    
    # Tabela 1301 - Área territorial brasileira
    url = "https://apisidra.ibge.gov.br/values/t/1301/n6/all/v/615"
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        data = response.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        
        # Limpar e processar
        df = df[['D1C', 'V']].copy()
        df.columns = ['codigo_ibge', 'area_km2']
        
        df['codigo_ibge'] = pd.to_numeric(df['codigo_ibge'], errors='coerce')
        df['area_km2'] = pd.to_numeric(df['area_km2'], errors='coerce')
        
        df = df.dropna()
        
        print(f"✅ Área de {len(df):,} municípios coletada!")
        print()
        print("📊 Estatísticas:")
        print(f"   Área total: {df['area_km2'].sum():,.0f} km²")
        print(f"   Maior município: {df['area_km2'].max():,.0f} km²")
        print(f"   Menor município: {df['area_km2'].min():,.2f} km²")
        
        return df
        
    except Exception as e:
        print(f"❌ Erro ao buscar área: {e}")
        return None

df_area = get_area()

if df_area is not None:
    print()
    print("📋 Amostra dos dados:")
    print(df_area.head(10))


🗺️ COLETANDO ÁREA TERRITORIAL

🔄 Buscando área territorial...
✅ Área de 5,565 municípios coletada!

📊 Estatísticas:
   Área total: 8,502,729 km²
   Maior município: 159,533 km²
   Menor município: 3.60 km²

📋 Amostra dos dados:
   codigo_ibge  area_km2
0      1100015    7067.0
1      1100023    4426.6
2      1100031    1314.4
3      1100049    3792.8
4      1100056    2783.3
5      1100064    1451.1
6      1100072    3060.3
7      1100080    4987.2
8      1100098    4518.0
9      1100106   24855.8


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 7: Dados complementares (manualmente ou via outras fontes)
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 INDICADORES COMPLEMENTARES")
print("="*80)
print()

print("ℹ️ NOTA IMPORTANTE:")
print()
print("Alguns indicadores não estão disponíveis via API SIDRA ou")
print("requerem processamento mais complexo. Para este projeto,")
print("usaremos dados simulados/exemplares para:")
print()
print("   • Taxa de alfabetização")
print("   • Mortalidade infantil")
print("   • Esgotamento sanitário adequado")
print("   • Estabelecimentos de saúde")
print()
print("💡 EM UM PROJETO REAL, você pode buscar em:")
print("   → Atlas do Desenvolvimento Humano: http://atlasbrasil.org.br")
print("   → DataSUS: https://datasus.saude.gov.br/")
print("   → SNIS: http://www.snis.gov.br/")
print()
print("Para fins didáticos, vamos simular dados realistas baseados")
print("em correlações conhecidas com população e PIB per capita.")
print()

def create_synthetic_indicators(df_base):
    """
    Cria indicadores sintéticos realistas para exemplo
    
    Args:
        df_base: DataFrame com população e PIB
    
    Retorna:
        DataFrame com indicadores adicionais
    """
    print("🔄 Gerando indicadores complementares...")
    
    df = df_base.copy()
    
    # Calcular PIB per capita
    df['pib_per_capita'] = (df['pib_milhoes'] * 1_000_000) / df['populacao']
    
    # Taxa de alfabetização (correlacionada com PIB per capita)
    # Municípios mais ricos tendem a ter maior alfabetização
    np.random.seed(42)
    base_alfabet = 85 + (df['pib_per_capita'] / df['pib_per_capita'].max()) * 10
    df['taxa_alfabetizacao'] = base_alfabet + np.random.normal(0, 2, len(df))
    df['taxa_alfabetizacao'] = df['taxa_alfabetizacao'].clip(70, 99.5)
    
    # Mortalidade infantil (inversamente correlacionada com PIB per capita)
    base_mortalidade = 20 - (df['pib_per_capita'] / df['pib_per_capita'].max()) * 10
    df['mortalidade_infantil'] = base_mortalidade + np.random.normal(0, 2, len(df))
    df['mortalidade_infantil'] = df['mortalidade_infantil'].clip(5, 30)
    
    # Esgotamento sanitário adequado
    base_esgoto = 60 + (df['pib_per_capita'] / df['pib_per_capita'].max()) * 30
    df['esgoto_adequado'] = base_esgoto + np.random.normal(0, 5, len(df))
    df['esgoto_adequado'] = df['esgoto_adequado'].clip(20, 95)
    
    # Estabelecimentos de saúde (proporcional à população)
    df['estabelecimentos_saude'] = (df['populacao'] / 5000) + np.random.poisson(2, len(df))
    df['estabelecimentos_saude'] = df['estabelecimentos_saude'].clip(1, None)
    
    print("✅ Indicadores criados!")
    print()
    print("📊 Resumo dos indicadores:")
    print(f"   Taxa alfabetização: {df['taxa_alfabetizacao'].min():.1f}% a {df['taxa_alfabetizacao'].max():.1f}%")
    print(f"   Mortalidade infantil: {df['mortalidade_infantil'].min():.1f} a {df['mortalidade_infantil'].max():.1f} por mil")
    print(f"   Esgoto adequado: {df['esgoto_adequado'].min():.1f}% a {df['esgoto_adequado'].max():.1f}%")
    print(f"   Estabelecimentos saúde: {df['estabelecimentos_saude'].min():.0f} a {df['estabelecimentos_saude'].max():.0f}")
    
    return df


📊 INDICADORES COMPLEMENTARES

ℹ️ NOTA IMPORTANTE:

Alguns indicadores não estão disponíveis via API SIDRA ou
requerem processamento mais complexo. Para este projeto,
usaremos dados simulados/exemplares para:

   • Taxa de alfabetização
   • Mortalidade infantil
   • Esgotamento sanitário adequado
   • Estabelecimentos de saúde

💡 EM UM PROJETO REAL, você pode buscar em:
   → Atlas do Desenvolvimento Humano: http://atlasbrasil.org.br
   → DataSUS: https://datasus.saude.gov.br/
   → SNIS: http://www.snis.gov.br/

Para fins didáticos, vamos simular dados realistas baseados
em correlações conhecidas com população e PIB per capita.



In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 8: Consolidar todos os dados
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🔗 CONSOLIDANDO TODOS OS DADOS")
print("="*80)
print()

print("🔄 Fazendo merge de todos os DataFrames...")

# Começar com municípios
df_final = df_municipios.copy()
print(f"   Base: {len(df_final):,} municípios")

# Merge com população
if df_populacao is not None:
    df_final = df_final.merge(df_populacao, on='codigo_ibge', how='left')
    print(f"   + População: {df_final['populacao'].notna().sum():,} registros")

# Merge com PIB
if df_pib is not None:
    df_final = df_final.merge(df_pib, on='codigo_ibge', how='left')
    print(f"   + PIB: {df_final['pib_milhoes'].notna().sum():,} registros")

# Merge com área
if df_area is not None:
    df_final = df_final.merge(df_area, on='codigo_ibge', how='left')
    print(f"   + Área: {df_final['area_km2'].notna().sum():,} registros")

# Remover linhas com dados essenciais faltantes
print()
print("🔄 Removendo municípios com dados incompletos...")
before = len(df_final)
df_final = df_final.dropna(subset=['populacao', 'pib_milhoes', 'area_km2'])
after = len(df_final)
print(f"   Removidos: {before - after:,} municípios")
print(f"   Restantes: {after:,} municípios")

# Criar indicadores derivados e complementares
print()
df_final = create_synthetic_indicators(df_final)

# Calcular densidade populacional
df_final['densidade'] = df_final['populacao'] / df_final['area_km2']

print()
print("✅ Dataset consolidado!")
print()
print("📊 Dimensões finais:")
print(f"   Linhas: {len(df_final):,}")
print(f"   Colunas: {len(df_final.columns)}")
print()
print("📋 Colunas disponíveis:")
for i, col in enumerate(df_final.columns, 1):
    print(f"   {i:2d}. {col}")


🔗 CONSOLIDANDO TODOS OS DADOS

🔄 Fazendo merge de todos os DataFrames...
   Base: 5,571 municípios
   + População: 5,571 registros
   + PIB: 5,570 registros
   + Área: 5,565 registros

🔄 Removendo municípios com dados incompletos...
   Removidos: 6 municípios
   Restantes: 5,565 municípios

🔄 Gerando indicadores complementares...
✅ Indicadores criados!

📊 Resumo dos indicadores:
   Taxa alfabetização: 79.0% a 94.4%
   Mortalidade infantil: 10.2 a 26.9 por mil
   Esgoto adequado: 43.1% a 88.2%
   Estabelecimentos saúde: 1 a 2387

✅ Dataset consolidado!

📊 Dimensões finais:
   Linhas: 5,565
   Colunas: 14

📋 Colunas disponíveis:
    1. codigo_ibge
    2. nome_municipio
    3. uf
    4. nome_uf
    5. regiao
    6. populacao
    7. pib_milhoes
    8. area_km2
    9. pib_per_capita
   10. taxa_alfabetizacao
   11. mortalidade_infantil
   12. esgoto_adequado
   13. estabelecimentos_saude
   14. densidade


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 9: Inspeção inicial dos dados
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🔍 INSPEÇÃO INICIAL DOS DADOS")
print("="*80)
print()

# Informações gerais
print("📊 INFORMAÇÕES GERAIS:")
print(df_final.info())

print("\n" + "="*70)
print("📈 ESTATÍSTICAS DESCRITIVAS:")
print("="*70)
print(df_final.describe())

print("\n" + "="*70)
print("🗺️ DISTRIBUIÇÃO POR REGIÃO:")
print("="*70)
print(df_final['regiao'].value_counts().to_frame('quantidade'))

print("\n" + "="*70)
print("🏆 TOP 10 MAIORES MUNICÍPIOS (POPULAÇÃO):")
print("="*70)
top_pop = df_final.nlargest(10, 'populacao')[['nome_municipio', 'uf', 'populacao', 'pib_milhoes']]
print(top_pop.to_string(index=False))

print("\n" + "="*70)
print("💰 TOP 10 MAIORES PIBs:")
print("="*70)
top_pib = df_final.nlargest(10, 'pib_milhoes')[['nome_municipio', 'uf', 'populacao', 'pib_milhoes']]
print(top_pib.to_string(index=False))

print("\n" + "="*70)
print("💎 TOP 10 MAIORES PIB PER CAPITA:")
print("="*70)
top_pib_pc = df_final.nlargest(10, 'pib_per_capita')[['nome_municipio', 'uf', 'populacao', 'pib_per_capita']]
print(top_pib_pc.to_string(index=False))


🔍 INSPEÇÃO INICIAL DOS DADOS

📊 INFORMAÇÕES GERAIS:
<class 'pandas.core.frame.DataFrame'>
Index: 5565 entries, 0 to 5570
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   codigo_ibge             5565 non-null   int64  
 1   nome_municipio          5565 non-null   object 
 2   uf                      5565 non-null   object 
 3   nome_uf                 5565 non-null   object 
 4   regiao                  5565 non-null   object 
 5   populacao               5565 non-null   int64  
 6   pib_milhoes             5565 non-null   float64
 7   area_km2                5565 non-null   float64
 8   pib_per_capita          5565 non-null   float64
 9   taxa_alfabetizacao      5565 non-null   float64
 10  mortalidade_infantil    5565 non-null   float64
 11  esgoto_adequado         5565 non-null   float64
 12  estabelecimentos_saude  5565 non-null   float64
 13  densidade               5565 non-null   float

In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 10: Salvar dados
# ═══════════════════════════════════════════════════════════════════════════

from pathlib import Path
from datetime import datetime
import json

print("\n" + "="*80)
print("💾 SALVANDO DADOS")
print("="*80)
print()

# Caminho base (sobe de notebooks/ para projeto_5/)
base_path = Path('..')

# Salvar CSV
output_path = base_path / 'data/raw/municipios_completo.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8')
print(f"✅ Dados salvos em: {output_path}")

# Salvar Excel também (mais fácil para visualizar)
output_excel = base_path / 'data/raw/municipios_completo.xlsx'
df_final.to_excel(output_excel, index=False, engine='openpyxl')
print(f"✅ Dados salvos em: {output_excel}")

# Salvar metadados
metadata = {
    'data_coleta': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_municipios': len(df_final),
    'n_colunas': len(df_final.columns),
    'colunas': list(df_final.columns),
    'ano_populacao': 2022,
    'ano_pib': 2021,
    'fontes': [
        'API IBGE Localidades',
        'API SIDRA - Tabela 6579 (População)',
        'API SIDRA - Tabela 5938 (PIB)',
        'API SIDRA - Tabela 1301 (Área)',
    ],
    'nota': 'Indicadores complementares são sintéticos para fins didáticos'
}

metadata_path = base_path / 'data/raw/metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✅ Metadados salvos em: {metadata_path}")

print()
print("="*80)
print("🎉 COLETA DE DADOS CONCLUÍDA!")
print("="*80)
print()
print("📊 RESUMO:")
print(f"   ✅ {len(df_final):,} municípios coletados")
print(f"   ✅ {len(df_final.columns)} variáveis por município")
print(f"   ✅ Dados salvos em data/raw/")
print()
print("👉 PRÓXIMO PASSO:")
print("   Abra o notebook: 02_analise_exploratoria.ipynb")
print()
print("="*80)



💾 SALVANDO DADOS

✅ Dados salvos em: ..\data\raw\municipios_completo.csv
✅ Dados salvos em: ..\data\raw\municipios_completo.xlsx
✅ Metadados salvos em: ..\data\raw\metadata.json

🎉 COLETA DE DADOS CONCLUÍDA!

📊 RESUMO:
   ✅ 5,565 municípios coletados
   ✅ 14 variáveis por município
   ✅ Dados salvos em data/raw/

👉 PRÓXIMO PASSO:
   Abra o notebook: 02_analise_exploratoria.ipynb



In [ ]:
df_final